# ME/CFS Research by Physiological System: Number of Publications

Self-contained notebook that produces a single chart — the **number of ME/CFS publications
per year** mentioning each of six physiological systems (2-day CPET / metabolic, immune,
autonomic, vascular & cardiac, gut microbiome, genetics), 1990 onwards, using the **PubMed
Entrez API**.

Each system is counted independently (title/abstract or MeSH keywords), scoped to ME/CFS
research. Categories overlap by design, so they do not sum to a total. Run all cells top to
bottom; the final cell renders the chart.

### 📦 1 | Install & Import

#### 1.1 | Install packages

In [ ]:
!pip install biopython >/dev/null
!pip install plotnine >/dev/null

#### 1.2 | Import packages

In [ ]:
from    Bio import Entrez
import  pandas as pd
from    datetime import datetime
import  matplotlib.pyplot as plt
from    plotnine import *
from    mizani.formatters import comma_format, percent_format
import  matplotlib.font_manager as fm
import  time

#### 1.3 | Load CrunchME brand font (Brutel)

Searches common Colab/Drive locations for `Brutel*.otf`; if none are found it prompts you to
upload them. Charts still render (default font) if it can't be found.

In [ ]:
BRAND_FONT = "Brutel"

# Search local or uploaded Brutel font files.
FONT_SEARCH_DIRS = [
    ".",
    "/content",
    "/content/fonts",
]

def _find_brutel(dirs):
    import glob, os
    hits = []
    for d in dirs:
        for ext in ("otf", "ttf"):
            hits += glob.glob(os.path.join(d, "**", f"Brutel*.{ext}"), recursive=True)
    return sorted(set(hits))

font_files = _find_brutel(FONT_SEARCH_DIRS)

# If nothing found and we are in Colab, offer an upload widget
if not font_files:
    try:
        from google.colab import files
        print("No Brutel font files found. Please upload them "
              "(select Brutel-Regular.otf and Brutel-Bold.otf, others optional):")
        files.upload()                 # saves into the current working dir
        font_files = _find_brutel(["."])
    except Exception as e:
        print("Automatic upload unavailable:", e)

# Register whatever we found with matplotlib
for fpath in font_files:
    try:
        fm.fontManager.addfont(fpath)
    except Exception as e:
        print("Could not register", fpath, "-", e)

# Activate the brand font if it is now available
_available = {f.name for f in fm.fontManager.ttflist}
if BRAND_FONT in _available:
    plt.rcParams["font.family"] = BRAND_FONT
    print(f"✓ Brand font '{BRAND_FONT}' active ({len(font_files)} file(s) loaded).")
else:
    print(f"⚠ '{BRAND_FONT}' not found — charts will use the default font. "
          f"Searched: {FONT_SEARCH_DIRS}")

### 🔧 2 | Parameters

#### 2.1 | Physiological systems

In [ ]:
# The six physiological systems to compare (order controls legend / colour order).
# "Total ME/CFS" is the reference series = all ME/CFS papers, regardless of system.
system_list = [
    "Metabolic / Energy",
    "Immune",
    "Autonomic",
    "Vascular & Cardiac",
    "Gut Microbiome",
    "Genetics",
]

# Full list including the reference baseline (used for the absolute-count plot)
series_list = system_list + ["Total ME/CFS"]

#### 2.2 | Colours

In [ ]:
# Colours loosely follow the emoji/theme colours used in the CrunchME deck.
# "Total ME/CFS" is a neutral grey so it reads as a reference baseline.
system_colors = {
    "Metabolic / Energy":  "#FFE066",   # 🔋 pastel yellow
    "Immune":              "#74C0FC",   # 🛡 blue
    "Autonomic":           "#FFA8A8",   # 🧠 pink
    "Vascular & Cardiac":  "#FF6B6B",   # 🫀 red
    "Gut Microbiome":      "#8CE99A",   # 🦠 green (stronger)
    "Genetics":            "#D0BFFF",   # 🧬 purple
    "Total ME/CFS":        "#CED4DA",   # ⚪ grey (reference baseline)
}

#### 2.3 | Time period

In [ ]:
# Define the start and end years for the PubMed API search.
# 1990 is used as the start: "chronic fatigue syndrome" was officially defined by the CDC
# in 1988 (Holmes criteria) and given its own MeSH index term then, so the modern ME/CFS
# literature effectively begins ~1988-1990. Before that the field had no agreed name/index.
start_year = 1990
end_year   = 2025   # NB: the current year will be partial until it ends

#### 2.4 | Smoothing (moving average) — set MA_WINDOW = 1 for raw yearly values

In [ ]:
# Moving-average window (in years). 3 = 3-year moving average; 1 = raw yearly counts.
MA_WINDOW = 3

#### 2.5 | Search terms (MeSH + title/abstract keywords)

In [ ]:
# Define a helper function to build the 'MeSH OR tiab' queries
def build_mesh_tiab_query(mesh_terms=None, tiab_phrases=None, extra_clauses=None):
    """
    Build a PubMed query that matches either:
      - MeSH Terms (topic indexing), OR
      - Title/Abstract phrases ([tiab])
    Optionally add extra clauses (already field-tagged).
    """
    mesh_terms   = mesh_terms or []
    tiab_phrases = tiab_phrases or []
    extra_clauses = extra_clauses or []

    parts = []

    if mesh_terms:
        mesh_part = " OR ".join([f'"{t}"[MeSH Terms]' for t in mesh_terms])
        parts.append(f"({mesh_part})")

    if tiab_phrases:
        tiab_part = " OR ".join([f'"{p}"[tiab]' for p in tiab_phrases])
        parts.append(f"({tiab_part})")

    if extra_clauses:
        extra_part = " OR ".join(extra_clauses)
        parts.append(f"({extra_part})")

    return "(" + " OR ".join(parts) + ")" if parts else ""


# ------------------------------------------------------------------
# Base ME/CFS clause — every system query is scoped to this.
# ------------------------------------------------------------------
mecfs_query = build_mesh_tiab_query(
    mesh_terms=["Myalgic Encephalomyelitis", "Chronic Fatigue Syndrome"],
    tiab_phrases=[
        "ME/CFS",
        "CFS/ME",
        "myalgic encephalomyelitis",
        "chronic fatigue syndrome",
        "systemic exertion intolerance disease",
        "SEID",
    ],
)

# ------------------------------------------------------------------
# Physiological-system clauses (keyword lists derived from the deck).
# ------------------------------------------------------------------
system_defs = {

    # 1. 🔋 Metabolic Dysfunction & Impaired Energy Production
    "Metabolic / Energy": build_mesh_tiab_query(
        mesh_terms=[
            "Energy Metabolism", "Mitochondria", "Oxidative Phosphorylation",
            "Adenosine Triphosphate", "Metabolomics", "Exercise Test",
            "Oxygen Consumption", "Citric Acid Cycle", "Glycolysis",
        ],
        tiab_phrases=[
            "mitochondria", "mitochondrial", "oxidative phosphorylation",
            "ATP production", "energy metabolism", "energy production",
            "metabolomics", "metabolome", "metabolic dysfunction",
            "metabolic inflexibility", "bioenergetic", "bioenergetics",
            "aerobic metabolism", "anaerobic threshold",
            "CPET", "cardiopulmonary exercise test", "two-day CPET",
            "2-day CPET", "VO2 peak", "VO2 max", "VO2max", "oxygen consumption",
            "pyruvate dehydrogenase", "high-resolution respirometry",
            "muscle biopsy", "skeletal muscle metabolism",
        ],
    ),

    # 2. 🛡 Chronic Immune System Activation & Exhaustion
    "Immune": build_mesh_tiab_query(
        mesh_terms=[
            "Killer Cells, Natural", "Cytokines", "T-Lymphocytes",
            "CD8-Positive T-Lymphocytes", "Inflammation", "Autoimmunity",
            "Immunity, Innate", "Immune System Diseases", "Interferons",
            "Interleukins", "Tumor Necrosis Factors",
        ],
        tiab_phrases=[
            "natural killer", "NK cell", "NK cells", "NK cytotoxicity",
            "cytokine", "cytokines", "T cell", "T-cell", "T cells", "T-cells",
            "CD8", "CD4", "immune exhaustion", "T cell exhaustion",
            "immune activation", "immune dysfunction", "inflammation",
            "inflammatory", "pro-inflammatory", "autoimmune", "autoimmunity",
            "interleukin", "interferon", "cytotoxicity", "immunophenotype",
            "immunophenotyping",
        ],
    ),

    # 3. 🧠 Autonomic Dysfunction
    "Autonomic": build_mesh_tiab_query(
        mesh_terms=[
            "Autonomic Nervous System Diseases", "Orthostatic Intolerance",
            "Postural Orthostatic Tachycardia Syndrome", "Tilt-Table Test",
            "Baroreflex", "Hypotension, Orthostatic", "Primary Dysautonomias",
            "Heart Rate",
        ],
        tiab_phrases=[
            "autonomic", "autonomic dysfunction", "dysautonomia",
            "orthostatic intolerance", "orthostatic hypotension",
            "tilt table", "tilt-table", "head-up tilt",
            "POTS", "postural orthostatic tachycardia",
            "postural tachycardia syndrome",
            "heart rate variability", "HRV", "baroreflex",
            "cerebral blood flow", "neurally mediated hypotension",
            "sympathetic", "parasympathetic", "vagal tone",
        ],
    ),

    # 4. 🫀 Vascular & Cardiac System
    "Vascular & Cardiac": build_mesh_tiab_query(
        mesh_terms=[
            "Endothelium, Vascular", "Microcirculation", "Hemodynamics",
            "Cardiac Output", "Blood Volume", "Blood Circulation",
            "Erythrocyte Deformability", "Ventricular Function",
        ],
        tiab_phrases=[
            "vascular", "endothelial", "endothelium", "endothelial dysfunction",
            "microvascular", "microcirculation", "cardiac output",
            "preload failure", "pre-load failure", "venous return",
            "oxygen extraction", "invasive CPET", "iCPET", "hemodynamic",
            "haemodynamic", "blood volume", "hypovolemia", "hypovolaemia",
            "red blood cell deformability", "erythrocyte deformability",
            "blood flow", "cardiovascular",
        ],
    ),

    # 5. 🦠 Gut Dysbiosis & Gut Microbiome
    "Gut Microbiome": build_mesh_tiab_query(
        mesh_terms=[
            "Gastrointestinal Microbiome", "Dysbiosis", "Microbiota",
            "Fatty Acids, Volatile", "Feces",
        ],
        tiab_phrases=[
            "microbiome", "microbiota", "dysbiosis", "gut microbiome",
            "gut microbiota", "intestinal microbiome", "intestinal microbiota",
            "gut bacteria", "butyrate", "short-chain fatty acid",
            "short chain fatty acid", "SCFA", "Faecalibacterium", "Roseburia",
            "gut-immune", "metagenomic", "metagenomics", "fecal microbiota",
            "faecal microbiota", "gut barrier",
        ],
    ),

    # 6. 🧬 Genetics Signals & Risk Factors
    "Genetics": build_mesh_tiab_query(
        mesh_terms=[
            "Genome-Wide Association Study", "Polymorphism, Single Nucleotide",
            "Genetic Predisposition to Disease", "Genetic Loci", "Genomics",
            "Genetic Variation",
        ],
        tiab_phrases=[
            "genetic", "genetics", "genome", "genome-wide",
            "genome-wide association", "GWAS", "polymorphism", "SNP",
            "single nucleotide polymorphism", "genetic variant",
            "genetic variation", "heritability", "risk allele",
            "genetic predisposition", "genomic", "DecodeME",
        ],
    ),
}

# ------------------------------------------------------------------
# Combine: each system query is scoped to ME/CFS research.
# "Total ME/CFS" is the ME/CFS base clause on its own (the reference baseline).
# ------------------------------------------------------------------
search_terms = {}  # six physiological systems only
for label, sys_query in system_defs.items():
    search_terms[label] = f"({mecfs_query}) AND ({sys_query})"

# Quick sanity check: show one combined query
print(search_terms["Metabolic / Energy"][:400], "...")

### 🚰 3 | Data Import — PubMed Entrez API

In [ ]:
def fetch_pubmed_data_for_multiple_terms(search_terms, start_year, end_year, sleep=0.34):
    Entrez.email = "your.email@example.org"  # Replace with a contact email before running
    # Entrez.api_key = "YOUR_NCBI_API_KEY"   # optional: uncomment for higher rate limit

    publication_counts = {term: [] for term in search_terms}
    years = range(start_year, end_year + 1)

    for year in years:
        for series_label, query in search_terms.items():
            full_query = f"{query} AND ({year}[dp])"

            # Small retry loop to ride out transient NCBI hiccups
            for attempt in range(3):
                try:
                    handle = Entrez.esearch(db="pubmed", term=full_query, retmax=0)
                    record = Entrez.read(handle)
                    handle.close()
                    count = int(record["Count"])
                    break
                except Exception as e:
                    if attempt == 2:
                        print(f"  ! {series_label} {year}: {e} — recording 0")
                        count = 0
                    else:
                        time.sleep(1.0)

            publication_counts[series_label].append(count)
            time.sleep(sleep)

        print(f"{year} done")

    df = pd.DataFrame(publication_counts, index=years)
    df.index.name = "Year"
    return df

# Fetch the data (runs ~1 minute)
publications_df = fetch_pubmed_data_for_multiple_terms(search_terms, start_year, end_year)

# Check result
publications_df.tail()

### 🥋 4 | Data Wrangling

#### 4.1 | Reshape + moving average

In [ ]:
# Transform the wide DataFrame to long format
df_long = (
    publications_df
    .reset_index()
    .melt(id_vars="Year", var_name="System", value_name="Publications")
)

# --- Moving-average smoothing (MA_WINDOW = 1 leaves the raw yearly values) ---
df_long = df_long.sort_values(["System", "Year"])
def _smooth(s):
    return s.rolling(MA_WINDOW, center=True, min_periods=1).mean()
df_long["Publications_MA"] = df_long.groupby("System")["Publications"].transform(_smooth)

# Check result
df_long.tail()

#### 4.2 | Order legend by final-year counts

In [ ]:
# Order the legend so systems appear high -> low by their value in the FINAL year.
final_year = int(df_long["Year"].max())

def order_by_final_year(df, value_col):
    order = (
        df[df["Year"] == final_year]
        .sort_values(value_col, ascending=False)["System"]
        .astype(str).tolist()
    )
    df = df.copy()
    df["System"] = pd.Categorical(df["System"].astype(str), categories=order, ordered=True)
    return df

df_abs = order_by_final_year(df_long[df_long["System"].isin(system_list)].copy(), "Publications_MA")
df_abs.head()

### 📈 5 | Chart — Number of Publications

In [ ]:
plot_abs = (
    ggplot(df_abs, aes(x="Year", y="Publications_MA", color="System"))
    + geom_line(size=1.5)
    + labs(
        title="ME/CFS research by physiological system: number of publications",
        subtitle=(f"{MA_WINDOW}-year moving average" if MA_WINDOW > 1 else "raw annual counts"),
        x="Year",
        y="Number of Publications",
    )
    + theme_classic()
    + theme(
        figure_size=(12, 8),
        text=element_text(family=BRAND_FONT),
        plot_title=element_text(size=22, weight="bold", color="white", hjust=0, family=BRAND_FONT),
        plot_subtitle=element_text(size=22, color="#ADB5BD", hjust=0, family=BRAND_FONT),
        axis_title=element_text(size=16, weight="bold", color="white", family=BRAND_FONT),
        axis_text=element_text(size=16, color="white", family=BRAND_FONT),
        legend_title=element_text(size=20, weight="bold", color="white", family=BRAND_FONT),
        legend_text=element_text(size=20, color="white", family=BRAND_FONT),
        legend_position=(0.05, 0.95),

        plot_background=element_rect(fill="#121421", color=None),
        panel_background=element_rect(fill="#121421", color=None),
        legend_background=element_rect(fill="#121421", color=None),
        legend_key=element_rect(fill="#121421", color=None),

        axis_line=element_line(color="white", size=1),
        axis_ticks=element_line(color="white", size=1),
        panel_grid_major_y=element_line(color="white", size=0.5, alpha=0.25),
        panel_grid_minor_y=element_blank(),
        panel_grid_major_x=element_blank(),
        panel_grid_minor_x=element_blank(),
    )
    + scale_y_continuous(labels=comma_format())
    + scale_x_continuous(limits=[start_year, end_year])
    + scale_color_manual(values=system_colors)
)

plot_abs